# 01 Reproduce Summer Baseline

**Date:** 2026-10-15  
**Author:** Xincheng (George) Ding  
**Branch:** `george`  
**Mode:** fully offline (no network, no database, no API keys)

**Purpose:** Load the summer prototype's graph-ready CSVs, profile them, check the Aura Free budget, run basic data-quality checks, and build the graph in memory with NetworkX to test the first queries before any Neo4j work.

**Input:** `IATI/reference/iati-graph-prototype/data/graph/*.csv` (read-only reference clone)  
**Output:** `fa26/data/interim/01_baseline_profile_2026-10-15.csv`

**Environment:** Anaconda base (pandas, networkx). Neo4j loading moves to a later notebook.

## 1. Setup

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "benchmark" / "manifest.json").exists())
FA26 = ROOT / "fa26"
sys.path.insert(0, str(ROOT / "benchmark"))
from common import verify_snapshot
SNAPSHOT_METADATA = verify_snapshot()
GRAPH_DIR = ROOT / "benchmark" / "snapshot"
INTERIM = FA26 / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

import pandas as pd
import networkx as nx


## 2. Load all node and relationship CSVs

In [ ]:
tables = {p.stem: pd.read_csv(p, dtype=str, keep_default_na=False) for p in sorted(GRAPH_DIR.glob("*.csv"))}

profile = pd.DataFrame(
    [{"table": k, "kind": "node" if k.startswith("nodes_") else "relationship",
      "rows": len(v), "columns": ", ".join(v.columns)} for k, v in tables.items()]
).sort_values(["kind", "rows"], ascending=[True, False])
profile

## 3. Aura Free budget (200k nodes / 400k relationships)

In [ ]:
AURA_NODES, AURA_RELS = 200_000, 400_000
n_nodes = profile.query("kind == 'node'")["rows"].sum()
n_rels = profile.query("kind == 'relationship'")["rows"].sum()
# activities also link to their transactions and budgets via iati_identifier
n_rels += len(tables["nodes_transactions"]) + len(tables["nodes_budgets"])
print(f"Nodes: {n_nodes:,} ({n_nodes / AURA_NODES:.1%} of cap)")
print(f"Relationships (approx): {n_rels:,} ({n_rels / AURA_RELS:.1%} of cap)")

## 4. Data-quality checks

In [ ]:
PK = {
    "nodes_activities": "iati_identifier", "nodes_organisations": "ref", "nodes_countries": "code",
    "nodes_sectors": "sector_key", "nodes_locations": "location_id", "nodes_transactions": "tx_id",
    "nodes_budgets": "budget_id", "nodes_sdgs": "sdg_key",
}
dupes = {t: int(tables[t][k].duplicated().sum()) for t, k in PK.items()}
blank = {t: int((tables[t][k] == "").sum()) for t, k in PK.items()}
pd.DataFrame({"duplicate_keys": dupes, "blank_keys": blank})

In [ ]:
# Orphan check: relationship rows pointing at nodes that do not exist
acts = set(tables["nodes_activities"]["iati_identifier"])
orgs = set(tables["nodes_organisations"]["ref"])
checks = {
    "rel_reports -> missing org": (~tables["rel_reports"]["org_ref"].isin(orgs)).sum(),
    "rel_reports -> missing activity": (~tables["rel_reports"]["iati_identifier"].isin(acts)).sum(),
    "rel_participates -> missing org": (~tables["rel_participates"]["org_ref"].isin(orgs)).sum(),
    "rel_targets_country -> missing country": (~tables["rel_targets_country"]["country_code"].isin(set(tables["nodes_countries"]["code"]))).sum(),
    "rel_classified_as -> missing sector": (~tables["rel_classified_as"]["sector_key"].isin(set(tables["nodes_sectors"]["sector_key"]))).sum(),
    "transactions -> missing activity": (~tables["nodes_transactions"]["iati_identifier"].isin(acts)).sum(),
    "rel_transaction_edges -> missing org": (~tables["rel_transaction_edges"]["org_ref"].isin(orgs)).sum(),
}
pd.Series({k: int(v) for k, v in checks.items()}, name="orphan_rows")

In [ ]:
tx = tables["nodes_transactions"].copy()
tx["value_num"] = pd.to_numeric(tx["value"], errors="coerce")
print("Activities per publisher:\n", tables["nodes_activities"]["publisher"].value_counts(), "\n")
print("Transaction currency:\n", tx["currency"].replace("", "(blank)").value_counts(), "\n")
print("Transaction type:\n", tx["type"].value_counts(), "\n")
print(f"Negative transactions: {(tx['value_num'] < 0).mean():.1%}")
print(f"Unparseable values: {tx['value_num'].isna().sum()}")
print("\nActivity match_basis (hunger filter):\n", tables["nodes_activities"]["match_basis"].value_counts())

In [ ]:
out = INTERIM / "01_baseline_profile_2026-10-15.csv"
profile.to_csv(out, index=False)
print("Saved", out)

## 5. Build the graph in memory (NetworkX, offline)

Same node labels and relationship types as the planned Neo4j schema, so the queries below translate directly to Cypher later.

In [ ]:
import networkx as nx

NODES = {  # csv -> (label, key)
    "nodes_activities": ("Activity", "iati_identifier"), "nodes_organisations": ("Organisation", "ref"),
    "nodes_countries": ("Country", "code"), "nodes_sectors": ("Sector", "sector_key"),
    "nodes_locations": ("Location", "location_id"), "nodes_transactions": ("Transaction", "tx_id"),
    "nodes_budgets": ("Budget", "budget_id"), "nodes_sdgs": ("SDG", "sdg_key"),
}
RELS = [  # csv, type, (from label, from col), (to label, to col)
    ("rel_reports", "REPORTS", ("Organisation", "org_ref"), ("Activity", "iati_identifier")),
    ("rel_participates", "PARTICIPATES_IN", ("Organisation", "org_ref"), ("Activity", "iati_identifier")),
    ("rel_targets_country", "TARGETS", ("Activity", "iati_identifier"), ("Country", "country_code")),
    ("rel_classified_as", "CLASSIFIED_AS", ("Activity", "iati_identifier"), ("Sector", "sector_key")),
    ("rel_located_at", "LOCATED_AT", ("Activity", "iati_identifier"), ("Location", "location_id")),
    ("rel_contributes_to", "CONTRIBUTES_TO", ("Activity", "iati_identifier"), ("SDG", "sdg_key")),
    ("nodes_transactions", "HAS_TRANSACTION", ("Activity", "iati_identifier"), ("Transaction", "tx_id")),
    ("nodes_budgets", "HAS_BUDGET", ("Activity", "iati_identifier"), ("Budget", "budget_id")),
]

G = nx.MultiDiGraph()
for csv, (label, key) in NODES.items():
    for r in tables[csv].to_dict("records"):
        G.add_node((label, r[key]), label=label, **r)
for csv, rtype, (fl, fc), (tl, tc) in RELS:
    df = tables[csv]
    props = [c for c in df.columns if c not in (fc, tc)] if csv.startswith("rel_") else []
    for source_index, r in zip(df.index, df.to_dict("records")):
        G.add_edge((fl, r[fc]), (tl, r[tc]), key=f"{csv}:row:{source_index + 1}", type=rtype, **{k: r[k] for k in props})

print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

## 6. First queries (Cypher equivalent shown above each)

In [ ]:
def out_nbrs(node, rtype):
    return [v for _, v, d in G.out_edges(node, data=True) if d["type"] == rtype]

def in_nbrs(node, rtype):
    return [u for u, _, d in G.in_edges(node, data=True) if d["type"] == rtype]

# MATCH (o:Organisation)-[:REPORTS]->(a:Activity) RETURN o.name, count(a) ORDER BY count(a) DESC
q1 = pd.DataFrame(
    [{"org": G.nodes[n].get("name"), "activities": len(out_nbrs(n, "REPORTS"))}
     for n, d in G.nodes(data=True) if d["label"] == "Organisation" and out_nbrs(n, "REPORTS")]
).sort_values("activities", ascending=False)
q1

In [ ]:
# MATCH (a:Activity)-[:TARGETS]->(:Country {code:'SD'}) RETURN a.publisher, a.title LIMIT 10
q2 = pd.DataFrame(
    [{"publisher": G.nodes[a]["publisher"], "title": G.nodes[a]["title"]}
     for a in in_nbrs(("Country", "SD"), "TARGETS")]
)
print(len(q2), "activities target Sudan")
q2.head(10)

In [ ]:
# MATCH (n) WHERE NOT (n)--() RETURN labels(n)[0], count(n)
isolated = pd.Series([G.nodes[n]["label"] for n in nx.isolates(G)], dtype=str).value_counts()
isolated if len(isolated) else "No isolated nodes"

## Reviewed integration notes
The row-preserving graph has 27,279 nodes and 32,291 relationships before transaction provider/receiver edges. Notebook 02 adds 26,830 source records, reaching 59,121. The original notebook collapsed 807 participation records. See analysis/GEORGE_REVIEW.md for reproduced evidence and limitations.
